In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"

trips_raw = spark.table(f"{CATALOG}.{SCHEMA}.trips_raw")
zones_raw = spark.table(f"{CATALOG}.{SCHEMA}.zone_lookup_raw")

print(f"Bronze trip rows: {trips_raw.count():,}")


In [0]:
valid_zone_ids = [row.LocationID for row in zones_raw.select("LocationID").distinct().collect()]

trips_clean = (
    trips_raw
    .dropDuplicates()
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter(F.col("trip_duration_minutes") <= 240)
    .filter(F.col("PULocationID").isin(valid_zone_ids))
    .filter(F.col("DOLocationID").isin(valid_zone_ids))
    .withColumn("trip_distance", F.col("trip_distance").cast("double"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("tip_amount", F.col("tip_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("_cleaned_at", F.current_timestamp())
)

rows_before = trips_raw.count()
rows_after = trips_clean.count()
print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning:  {rows_after:,}")
print(f"Rows dropped:         {rows_before - rows_after:,} ({(1 - rows_after / rows_before):.1%})")


In [0]:
(
    trips_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.trips_clean")
)

zones_clean = (
    zones_raw
    .withColumnRenamed("LocationID", "location_id")
    .withColumnRenamed("Borough", "borough")
    .withColumnRenamed("Zone", "zone")
    .filter(F.col("borough").isNotNull())
    .dropDuplicates(["location_id"])
)

(
    zones_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.zones_clean")
)

print("Silver cleaning complete.")